In [230]:
import numpy as np
from scipy.stats import norm, kurtosis
from scipy.stats import skew

In [265]:
%ls

baci_fem_numba.ipynb             RMD_ckeck.ipynb
contact_area.dat                 rough_surface_grid_65.dat
force_rough_surface_grid_65.dat  rough_surface_grid_9.dat
force_rough_surface_grid_9.dat   simulation_output.dat
import_data.ipynb                sup2.dat
import_dat.ipynb                 sup5.dat
plot_contour.ipynb               weirstrass-mandelbrot.ipynb
random_postprocess_theory.ipynb


In [232]:
%pwd

'/home/a11btasa/deep_sim/jupyter_notebook'

In [233]:
size_surf = 33

In [234]:
z = np.loadtxt("sup5.dat", delimiter=";",usecols=range(size_surf))
z.mean()

52.794441103122125

In [235]:
delta = 1/size_surf*1000

In [236]:
lato = 1000
n_ele = size_surf

x_lin = np.linspace(lato/n_ele,lato,n_ele)
y_lin = np.linspace(lato/n_ele,lato,n_ele)

y, x = np.meshgrid(x_lin, y_lin)

# Compute profile statistics: slopes, maxima 2D (peaks) curvatures

In [237]:
slope_x, slope_y = np.gradient(z,delta) #slopes

In [238]:
slope_x

array([[-0.04408239, -0.03565241,  0.39349771, ...,  0.19487252,
         0.62138406, -0.45595595],
       [-0.28027779,  0.17361963,  0.08469899, ..., -0.01296979,
         0.46051262, -0.17282953],
       [ 0.47356818, -0.14306998,  0.07410591, ...,  0.20877292,
         0.03021462,  0.20469837],
       ...,
       [ 0.07404228,  0.33326989,  0.18172783, ...,  0.10640404,
        -0.43232607, -0.28010535],
       [ 0.33789046, -0.00603367,  0.19062384, ...,  0.35034052,
         0.03398848, -0.37519211],
       [ 0.27385314, -0.09021052, -0.06708042, ...,  0.0692277 ,
         0.34595698, -0.30127664]])

In [239]:
slope_x_2 = slope_x[1:-1,1:-1]
#slope_x_2

In [240]:
slope_y

array([[ 0.07371177, -0.11173985,  0.02023012, ..., -0.51639496,
        -0.07599585,  0.35333882],
       [ 0.08214175,  0.1070502 , -0.03462106, ...,  0.32130872,
        -0.40141009, -0.72400119],
       [ 0.98150663,  0.25323693, -0.27581469, ...,  0.27910118,
        -0.23585559, -0.91334549],
       ...,
       [ 0.21527972,  0.05001186,  0.24361088, ...,  0.14825334,
         0.44347624,  0.56156179],
       [-0.10850489,  0.07321202,  0.27464951, ..., -0.13843216,
        -0.09680423,  0.39043422],
       [-0.47256855, -0.09725476, -0.05245987, ..., -0.02041771,
        -0.2820564 , -0.2567994 ]])

In [241]:
slope_y_2 = slope_y[1:-1,1:-1]
# slope_y_2

## Evaluate 2D maxima (peaks)

In [242]:
n_peaks = 0
curv_peak = []
z_peak = []
for j in range(1,size_surf-1):
    for i in range(1,size_surf-1):
        if z[i, j] > z[i, j-1] and z[i,j]>z[i,j+1]:
            n_peaks += 1
            curv_peak.append(-(z[i,j+1]-2*z[i,j]+z[i,j-1])/(delta**2))
            z_peak.append(z[i,j])

## Statistics of the slopes and peaks

In [262]:
# statistics of z
m0 = np.std(z, ddof =1)

# statistics profile slopes
rms_slopex = np.std(slope_x_2 ,ddof =1)
rms_slopey = np.std(slope_y_2, ddof =1)

m2x = rms_slopex**2
m2y = rms_slopey**2


# statistics heights of peaks
mean_z_peaks = np.mean(z_peak)
rms_z_peaks = np.std(z_peak, ddof =1)
ks_z_peaks = kurtosis(z_peak, fisher=False)
sk_z_peaks = skew(z_peak)

# statistics curvatures of peaks
mean_curv_peak = np.mean(curv_peak)
rms_curv_peak = np.std(curv_peak, ddof =1)
ks_curv_peak = kurtosis(curv_peak)
sk_curv_peak = skew(curv_peak)

m4 = rms_curv_peak**2

density_peaks = n_peaks/(size_surf*size_surf) # density of peaks

alfa_x = m0*m4/m2x**2
alfa_y = m0*m4/m2y**2

# Compute the asperity (3D maxima) heights, curvatures and statistics

In [244]:
Curvx = np.zeros((size_surf-2,size_surf-2))
Curvy = np.zeros((size_surf-2,size_surf-2))


for i in range(1,size_surf-1):
   for j in range(1,size_surf-1):
       if z[i,j]>z[i,j-1] and z[i,j]>z[i,j+1]:
           Curvy[i-1,j-1]=-2*(-delta*z[i,j-1]+2*delta*z[i,j]-delta*z[i,j+1])/(-delta*y[j,j-1]**2+2*delta*y[j,j]**2-delta*y[j,j+1]**2)

for i in range(1,size_surf-1):
   for j in range(1,size_surf-1):
       if z[i,j]>z[i-1,j] and z[i,j]>z[i+1,j]:
           Curvx[i-1,j-1]=-2*(-delta*z[i-1,j]+2*delta*z[i,j]-delta*z[i+1,j])/(-delta*x[j-1,j]**2+2*delta*x[j,j]**2-delta*x[j+1,j]**2)


In [245]:
mask_vector = Curvx*Curvy
mask_crierion = mask_vector!= 0
curv = np.sqrt(mask_vector[mask_crierion])
z_without_border = z[1:-1,1:-1]
H = z_without_border[mask_crierion]

print(f"mean curv: {curv.mean()}")
print(f"mean H: {H.mean()}")

mean curv: 0.030183618781952547
mean H: 66.25531305755396


## Statistics of the asperity (3D maxima) heights and curvatures

In [264]:
# statistics of asperity (3D maxima) heights
mean_z_asperities = np.mean(H)
rms_z_asperities = np.std(H, ddof=1)
ks_z_asperities = kurtosis(H)
sk_z_asperities = skew(H)


# statistics of asperity (3D maxima) curvatures
mean_curv_asperities = np.mean(curv)
rms_curv_asperities = np.std(curv, ddof=1)
ks_curv_asperities = kurtosis(curv, fisher=False)
sk_curv_asperities = skew(curv)

density_asperities = H.shape[0]/(size_surf*size_surf) # density of peaks
